# Constrained Tikhonov regularization implemented by (accelerated) forward-backward splitting
We consider the two-dimensional deconvolution problems to find a non-negative function f given data 
$$
    d \sim \mathrm{Pois}(h*f)
$$
with a non-negative convolution kernel $h$, and $\mathrm{Pois}$ denotes the element-wise Poisson distribution.

We explore the use of the semismooth Newton method to implement constrained Tikhonov regularization 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\| h*f-d\|^2_{L^2(w)} + \alpha \|f\|^2_{L^2}\right]
$$
with a weight $w = \frac{1}{\sqrt{d+1}}$. The regularization parameter $\alpha$ is chosen by the discrepancy principle.   

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mplib

from regpy.operators.convolution import GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import TikhonovRegularizationSetting, RegularizationSetting
from regpy.solvers.linear.semismoothNewton import SemismoothNewton_nonneg
from regpy.solvers.linear.proximal_gradient import ForwardBackwardSplitting, FISTA
from regpy.solvers.linear.primal_dual import PDHG
from regpy.hilbert import L2
from regpy.stoprules import DualityGapStopping
from regpy.functionals import QuadraticLowerBound, QuadraticBilateralConstraints

from test_images import mixed
from comparison_plot import comparison_plot

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

### creating Poisson distributed synthetic data

In [ ]:
grid, exact_sol = mixed(M=256,N=256,fac=200.)
r"""grid is the underlying UniformGridFcts vector space, and exact_sol the exact solution."""
a=0.15
conv =  GaussianBlur(grid,a,pad_amount=16)
r"""Convolution operator $f\mapsto h*f$ for the convolution kernel $h(x)=\exp(-|x|_2^2/a^2)$."""
blur = conv(exact_sol)
blur[blur<0] = 0.
"""Simulated exact data."""
data = np.random.poisson(blur)
"""Simulated measured data. The Poisson distribution occurs if photon count detectors are used."""
comparison_plot(grid,exact_sol,data,title_left='noisy measurement data')

## Semismooth Newton method
So far we just run one step. TODO: More comparisons, e.g. by computation time. 

In [ ]:
weighted_data_space = L2(grid, weights = 1./(1.+data))
setting = RegularizationSetting(op=conv, penalty=L2, data_fid=weighted_data_space)
alpha = 0.001

SSNewton_nn = SemismoothNewton_nonneg(setting, data, alpha, TOL = 0.01, 
                                    logging_level=logging.DEBUG, cg_logging_level=logging.INFO)
it = iter(SSNewton_nn)

# Reconstructions with one sided constraints

In [ ]:
penLower = QuadraticLowerBound(grid,x0=0,lb=0)
alpha = 1e-3
n_iter = 400 
settingLower = TikhonovRegularizationSetting(op=conv, penalty=penLower, data_fid = L2,data_fid_shift=data,regpar=alpha)

### Forward-backward splitting

In [ ]:
FB_solver_lb = ForwardBackwardSplitting(settingLower)
stop_FB_lb=DualityGapStopping(FB_solver_lb,threshold = 1., max_iter=n_iter,logging_level=logging.WARNING)
FB_solver_lb.run(stoprule=stop_FB_lb)

comparison_plot(grid,exact_sol,FB_solver_lb.x,title_left='Forward-backward')

### FISTA

In [ ]:
FISTA_solver_lb = FISTA(settingLower)
stop_FISTA_lb=DualityGapStopping(FISTA_solver_lb,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
FISTA_solver_lb.run(stoprule=stop_FISTA_lb)
    
comparison_plot(grid,exact_sol,FISTA_solver_lb.x,title_left='FISTA reco')

### Primal-Dual Hybrid Gradient 

In [ ]:
PDHG_solver_lb = PDHG(settingLower)
stop_PDHG_lb=DualityGapStopping(PDHG_solver_lb,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_solver_lb.run(stoprule=stop_PDHG_lb)

comparison_plot(grid,exact_sol,PDHG_solver_lb.x, title_left='PDHG reco')

In [ ]:
plt.semilogy(stop_FB_lb.gap_stat,label='ForwardBackward')
plt.semilogy(stop_FISTA_lb.gap_stat,label='FISTA')
plt.semilogy(stop_PDHG_lb.gap_stat,label='PDHG')
plt.legend()
plt.xlabel('it. step'); plt.ylabel('duality gap')
plt.title('convergence for nonnegativity constraint')

# Reconstructions for bilateral constraints

### define setting

In [ ]:
pen = QuadraticBilateralConstraints(grid,lb=0, ub=400,eps=1e-14)
alpha = 1e-4
n_iter = 1000 
settingBilateral = TikhonovRegularizationSetting(op=conv, penalty=pen, data_fid = weighted_data_space,data_fid_shift=data,regpar=alpha)

### Forward-backward splitting

In [ ]:
FB_solver_bil = ForwardBackwardSplitting(settingBilateral)
stop_FB_bil = DualityGapStopping(FB_solver_bil,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
FB_solver_bil.run(stoprule=stop_FB_bil)

comparison_plot(grid,exact_sol,FB_solver_bil.x,title_left='Forward-backward')

### FISTA

In [ ]:
FISTA_solver_bil = FISTA(settingBilateral)
stop_FISTA_bil = DualityGapStopping(FISTA_solver_bil,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
gap_FISTA_bil = np.zeros((stop_FISTA_bil.max_iter+1,))
gap_FISTA_bil[0] = FISTA_solver_bil.gap
for step_nr, (x,y)  in enumerate(FISTA_solver_bil.while_(stop_FISTA_bil)):
    gap_FISTA_bil[step_nr+1] = FISTA_solver_bil.gap
comparison_plot(grid,exact_sol,x,title_left='FISTA reco')

### Primal-dual hybrid gradient (Chambolle-Pock) methods

In [ ]:
PDHG_solver_bil = PDHG(settingBilateral)
stop_PDHG_bil = DualityGapStopping(PDHG_solver_bil,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_solver_bil.run(stoprule=stop_PDHG_bil)

comparison_plot(grid,exact_sol,PDHG_solver_bil.x,title_left='PDHG reco')

### Convergence plots

In [ ]:
plt.semilogy(stop_FB_bil.gap_stat,label='ForwardBackward')
plt.semilogy(stop_FISTA_bil.gap_stat,label='FISTA')
plt.semilogy(stop_PDHG_bil.gap_stat,label='PDHG')
plt.legend()
plt.xlabel('it. step'); plt.ylabel('duality gap')
plt.title('convergence for bilateral constraints')